# Performance Evaluation (Colab)

This notebook evaluates saved poker agents by loading trained models from Google Drive and running a small round-robin tournament. It is optimized for execution in Google Colab.

## Setup

The cells below prepare the environment, mount Google Drive (when running in Colab), and make the repository available on `sys.path`.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore[attr-defined]
    IN_COLAB = True
except Exception:  # pragma: no cover - best effort detection
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/texas_holdem_ai_gatc")
else:
    REPO_ROOT = Path.cwd()

if REPO_ROOT.exists():
    os.chdir(REPO_ROOT)
    repo_src = REPO_ROOT / "src"
    if repo_src.exists() and str(repo_src) not in sys.path:
        sys.path.insert(0, str(repo_src))
else:
    raise FileNotFoundError(f"Expected repository at {REPO_ROOT}.")

print(f"Running in Colab: {IN_COLAB}")
print(f"Repository root: {REPO_ROOT}")


In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    print("Google Drive mounted.")
else:
    print("Google Colab not detected; skipping Drive mount.")


## Load models from Google Drive

Update `GOOGLE_DRIVE_MODELS_DIR` if your models live in a different folder. The code mirrors the models into the repository's `models/` directory so that the evaluation utilities can discover them.

In [ ]:
import shutil

GOOGLE_DRIVE_MODELS_DIR = Path("/content/drive/MyDrive/texas_holdem/models")
LOCAL_MODELS_DIR = REPO_ROOT / "models"
LOCAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and not GOOGLE_DRIVE_MODELS_DIR.exists():
    raise FileNotFoundError(
        "Could not find models in Google Drive. Set GOOGLE_DRIVE_MODELS_DIR to the correct path."
    )

copied = []
for src in GOOGLE_DRIVE_MODELS_DIR.glob("*.pt") if GOOGLE_DRIVE_MODELS_DIR.exists() else []:
    dst = LOCAL_MODELS_DIR / src.name
    if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
        shutil.copy2(src, dst)
        copied.append(dst.name)

if copied:
    print("Copied models:")
    for name in copied:
        print(f"  - {name}")
else:
    available = list(LOCAL_MODELS_DIR.glob("*.pt"))
    if available:
        print("Models already present locally. No copies needed.")
    else:
        raise FileNotFoundError(
            "No models found. Upload *.pt files to the Google Drive directory before running this cell."
        )


## Inspect available models

In [ ]:
sorted_models = sorted(LOCAL_MODELS_DIR.glob("*.pt"))
if not sorted_models:
    raise RuntimeError("No local models were discovered; make sure the previous step succeeded.")

for path in sorted_models:
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{path.name}: {size_mb:.2f} MB")


## Run a tournament

Configure the number of games per match and select the device (`cpu` or `cuda`).

In [ ]:
from typing import Literal

import torch

from poker_ai.evaluation.performance_analysis import run_tournament

from pathlib import Path

device: Literal["cpu", "cuda"]
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

model_paths = [str(path) for path in sorted_models]
print(f"Running evaluation on {len(model_paths)} models using device='{device}'.")

results = run_tournament(model_paths, games_per_match=10, device=device)

print("\nTournament results (wins per model):")
for path, wins in sorted(results.items(), key=lambda item: item[1], reverse=True):
    print(f"{Path(path).name}: {wins}")
